In [1]:
# Importing from previous notebook
from google.colab import drive
drive.mount('/content/drive')
%run "/content/drive/MyDrive/Colab Notebooks/1_UnderstandingLeastSquareForCardinalComparison.ipynb"

Mounted at /content/drive
True scores (mu): [ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
Edges: [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4)]
(0, 1) : [0.4008415  2.21419127 1.40241318 0.16550407 1.1775385 ]
(1, 0) : [-0.4008415  -2.21419127 -1.40241318 -0.16550407 -1.1775385 ]
(0, 2) : [ 1.31140374  1.38774058 -2.5900802   0.45246684 -0.40201835]
(2, 0) : [-1.31140374 -1.38774058  2.5900802  -0.45246684  0.40201835]
(0, 3) : [-1.19018282  1.53317233 -0.63208268 -0.90409654]
(3, 0) : [ 1.19018282 -1.53317233  0.63208268  0.90409654]
(0, 4) : [0.21543187 0.13061368]
(4, 0) : [-0.21543187 -0.13061368]
(1, 2) : [ 0.16148698 -0.0067602  -1.8870506   0.34427535]
(2, 1) : [-0.16148698  0.0067602   1.8870506  -0.34427535]
(1, 3) : [-1.28817524]
(3, 1) : [1.28817524]
(1, 4) : [ 0.30475267 -1.86378105 -1.23229698  0.29275031]
(4, 1) : [-0.30475267  1.86378105  1.23229698 -0.29275031]
[[0. 5. 5. 4. 2.]
 [5. 0. 4. 1. 4.]
 [5. 4. 0. 0. 0.]
 [4. 1. 0. 0. 0.]
 [2. 4. 0. 0

# **Downloading Both IPL Datasets**

In [ ]:
#  Implementation of T20 cricket predition using context

import kagglehub
path = kagglehub.dataset_download("ashraykothari/ipldataset")
!ls {path}


# Load deliveries.csv into a DataFrame
deliveries_df = pd.read_csv(f"{path}/deliveries.csv")

# Load matches.csv into a DataFrame
matches_df = pd.read_csv(f"{path}/matches.csv")

print("Deliveries DataFrame head:")
display(deliveries_df)
print("\nMatches DataFrame head:")
display(matches_df)


# **Data Processing**

In [ ]:
group_deliveries = deliveries_df.groupby('match_id').sum(numeric_only=True)
merged_df = pd.merge(matches_df, group_deliveries, left_on='id', right_on='match_id', how='inner')
# merged_df= merged_df.dropna()

# Drop some columns
columns_to_drop = [
    'player_of_match',
    'season',
    'venue',
    'umpire1',
    'umpire2',
    'umpire3',
    'dl_applied',
    'inning',
    'result',
    'is_super_over'

]
merged_df = merged_df.drop(columns=columns_to_drop, errors='ignore')

print("Merged DataFrame head (matches and aggregated deliveries):")
display(merged_df)

# **Declaring Home Ground Sites**

In [ ]:
# Making home ground arrangement

city_and_team = {
    'Sunrisers Hyderabad': ['Hyderabad', 'Visakhapatnam', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Mumbai Indians': ['Mumbai', 'Port Elizabeth', 'Durban', 'Centurion', 'East London', 'Johannesburg', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Gujarat Lions': ['Ahmedabad', 'Rajkot', 'Kanpur'],
    'Rising Pune Supergiant': ['Pune', 'Indore'],
    'Royal Challengers Bangalore': ['Bangalore', 'Bengaluru', 'Cape Town', 'Port Elizabeth', 'Durban', 'Centurion', 'Johannesburg', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Kolkata Knight Riders': ['Kolkata', 'Cuttack', 'Cape Town', 'Durban', 'Centurion', 'Johannesburg', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Delhi Daredevils': ['Delhi', 'Raipur', 'Dharamsala', 'Cape Town', 'Durban', 'Centurion', 'Johannesburg', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Kings XI Punjab': ['Chandigarh', 'Mohali', 'Indore', 'Dharamsala', 'Cuttack', 'Cape Town', 'Durban', 'Centurion', 'Johannesburg', 'Kimberley', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Chennai Super Kings': ['Chennai', 'Ranchi', 'East London', 'Port Elizabeth', 'Durban', 'Centurion', 'Johannesburg', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Rajasthan Royals': ['Jaipur', 'Ahmedabad', 'Cape Town', 'Durban', 'Centurion', 'Johannesburg', 'Kimberley', 'Abu Dhabi', 'Sharjah', 'Dubai'],
    'Deccan Chargers': ['Hyderabad', 'Cuttack', 'Nagpur', 'Cape Town', 'Johannesburg', 'Durban', 'Centurion', 'Kimberley'],
    'Kochi Tuskers Kerala': ['Kochi', 'Indore'],
    'Pune Warriors': ['Pune', 'Cuttack', 'Navi Mumbai'],
    'Rising Pune Supergiants': ['Pune', 'Visakhapatnam'],
    'Delhi Capitals': ['Delhi', 'Visakhapatnam', 'Abu Dhabi', 'Sharjah', 'Dubai']
}





# **Home Ground Covariate**

In [ ]:
def is_home_ground(row):
    city = row['city']
    team1 = row['team1']
    team2 = row['team2']
    score = 0

    # Check if team1 has a home ground and if the match city is one of them
    if team1 in city_and_team and city in city_and_team[team1]:
        score += 1
    # Check if team2 has a home ground and if the match city is one of them
    if team2 in city_and_team and city in city_and_team[team2]:
        score -= 1
    return score


merged_df['home_ground_advantage'] = merged_df.apply(is_home_ground, axis=1)

print("DataFrame with new 'home_ground_advantage' column:")
display(merged_df)

# **Days From Previous Match Covariate**

In [ ]:
merged_df['date'] = pd.to_datetime(merged_df['date'], format='mixed', dayfirst=True)
merged_df = merged_df.sort_values(by='date').reset_index(drop=True)

# Initialize a dictionary to store the last played date for each team
all_teams = pd.concat([merged_df['team1'], merged_df['team2']]).unique()
last_played_date = {team: pd.NaT for team in all_teams}

# Lists to store the calculated days
days_from_prev_t1_matches = []
days_from_prev_t2_matches = []


for index, row in merged_df.iterrows():
    current_date = row['date']
    team1 = row['team1']
    team2 = row['team2']

    # Calculate days since team1 last played
    if pd.isna(last_played_date[team1]):
        days_since_t1 = 10  # Fresh team gets neutral maximum rest status
    else:
        days_since_t1 = (current_date - last_played_date[team1]).days
        if days_since_t1 > 200:
            days_since_t1 = 10 # Cross-season fresh team gets neutral maximum rest
    days_from_prev_t1_matches.append(days_since_t1)

    # Calculate days since team2 last played
    if pd.isna(last_played_date[team2]):
        days_since_t2 = 10
    else:
        days_since_t2 = (current_date - last_played_date[team2]).days
        if days_since_t2 > 200:
            days_since_t2 = 10
    days_from_prev_t2_matches.append(days_since_t2)

    last_played_date[team1] = current_date
    last_played_date[team2] = current_date

# Add the new columns to the DataFrame
merged_df['days_from_prev_t1_match'] = days_from_prev_t1_matches
merged_df['days_from_prev_t2_match'] = days_from_prev_t2_matches

# Calculate the difference
merged_df['days_since_last_match_diff'] = merged_df['days_from_prev_t1_match'] - merged_df['days_from_prev_t2_match']

print("DataFrame with new 'days_since_last_match_diff' column:")
display(merged_df)

# **Previous Match Outcome Advantage**

In [ ]:
last_match_outcome = {team: None for team in all_teams}

# List to store the calculated previous match advantage
previous_match_advantage_list = []

# Iterate through each row to calculate previous match advantage
for index, row in merged_df.iterrows():
    current_team1 = row['team1']
    current_team2 = row['team2']
    current_winner = row['winner']

    team1_last_outcome = last_match_outcome[current_team1]
    team2_last_outcome = last_match_outcome[current_team2]

    # Default score is 0 (both won, both lost, or one/both haven't played)
    score = 0

    if team1_last_outcome is True and team2_last_outcome is False:
        score = 1
    elif team1_last_outcome is False and team2_last_outcome is True:
        score = -1

    previous_match_advantage_list.append(score)

    # Update last played outcome for both teams
    last_match_outcome[current_team1] = (current_team1 == current_winner)
    last_match_outcome[current_team2] = (current_team2 == current_winner)


merged_df['previous_match_advantage'] = previous_match_advantage_list

print("DataFrame with new 'previous_match_advantage' column:")
display(merged_df)


# **Winner Encoded**

In [ ]:
merged_df['winner_encoded'] = np.where(
    merged_df['winner'] == merged_df['team1'], 1,
    np.where(merged_df['winner'] == merged_df['team2'], -1, 0)
)

print("DataFrame with new 'winner_encoded' column:")
display(merged_df)

# **Win By Runs to be used as margin for the winning team that is Team1 if positive and Team2 if negative and extrapolated accordingly**

In [ ]:
second_inning_df = deliveries_df[deliveries_df['inning'] == 2]

# Calculate total runs for the second inning per match
total_runs_inning2 = second_inning_df.groupby('match_id')['total_runs'].sum().reset_index()
total_runs_inning2 = total_runs_inning2.rename(columns={'total_runs': 'total_runs_inning2'})

# Find the last over and ball for the second inning per match
last_delivery_inning2 = second_inning_df.sort_values(by=['match_id', 'over', 'ball']).groupby('match_id').tail(1)
last_delivery_inning2 = last_delivery_inning2[['match_id', 'over', 'ball']].rename(columns={'over': 'last_over_inning2', 'ball': 'last_ball_inning2'})

# Merge the results
second_inning_stats = pd.merge(total_runs_inning2, last_delivery_inning2, on='match_id', how='left')
second_inning_stats['run_rate'] = second_inning_stats['total_runs_inning2'] / (second_inning_stats['last_over_inning2'] + second_inning_stats['last_ball_inning2'] / 6)
second_inning_stats['over_reamining'] = 20 - second_inning_stats['last_over_inning2'] + (6- second_inning_stats['last_ball_inning2'] )/ 6


# Before merging, remove any existing 'run_rate' related columns from merged_df
columns_to_drop_if_exist = [col for col in merged_df.columns if 'run_rate' in col or 'over_reamining' in col or 'remaining_overs_decimal' in col]
if columns_to_drop_if_exist:
    merged_df = merged_df.drop(columns=columns_to_drop_if_exist, errors='ignore')

# Merge second_inning_stats with merged_df on match_id
merged_df = pd.merge(merged_df, second_inning_stats[['match_id', 'run_rate', 'over_reamining']], left_on='id', right_on='match_id', how='left')


merged_df = merged_df.reset_index(drop=True)
merged_df = merged_df.rename(columns={'over_reamining': 'remaining_overs_decimal'})
merged_df = merged_df.drop(columns=['match_id'], errors='ignore')
merged_df['win_by_runs'] = merged_df['win_by_runs'].astype(float)
merged_df['year'] = merged_df['date'].dt.year

print("Merged DataFrame after adding second inning run rate and remaining overs:")
display(merged_df)

# Create a boolean mask for the condition
condition = (merged_df['win_by_wickets'] != 0) & (merged_df['win_by_runs'] == 0)

# Calculate the new values only for the rows that meet the condition
new_values_for_condition = -1*merged_df.loc[condition, 'run_rate'] * merged_df.loc[condition, 'remaining_overs_decimal']

# Assign these calculated values back to 'win_by_runs' using the same condition
merged_df.loc[condition, 'win_by_runs'] = new_values_for_condition
print("DataFrame after modifying 'win_by_runs' column:")

display(merged_df)

In [ ]:
# Count how many rows would be dropped by dropna() in each year
for year in sorted(merged_df['year'].unique()):
    year_df = merged_df[merged_df['year'] == year]
    total_rows = len(year_df)
    clean_rows = len(year_df.dropna())
    dropped_rows = total_rows - clean_rows
    print(f"Year {year}: Total matches = {total_rows}, Rows with NaN (to be dropped) = {dropped_rows}")

In [ ]:
merged_df = merged_df.dropna()
merged_df.columns

In [ ]:
columns_to_drop_from_merged_df = [
    'wide_runs',
    'bye_runs',
    'legbye_runs',
    'noball_runs',
    'penalty_runs',
    'batsman_runs',
    #'win_by_wickets', # Commented out to prevent dropping this column
    'days_from_prev_t1_match',
    'days_from_prev_t2_match',
    'last_over_inning2',
    'last_ball_inning2',
    'overs_bowled_decimal',
    'run_rate',
    'remaining_overs_decimal',
    'winner',
    'over',
    'ball',
    'total_runs'

]
merged_df = merged_df.drop(columns=columns_to_drop_from_merged_df, errors='ignore')

print("DataFrame after dropping specified columns:")
display(merged_df)

# **Toss Winner Encoding**

In [ ]:
def encode_toss_winner(row):
    if row['toss_winner'] == row['team1']:
        return 1
    elif row['toss_winner'] == row['team2']:
        return -1
    else:
        return 0 # Should not happen if toss_winner is always team1 or team2

merged_df['toss_winner_encoded'] = merged_df.apply(encode_toss_winner, axis=1)

print("DataFrame with new 'toss_winner_encoded' column:")
display(merged_df)

In [ ]:
columns_to_drop_from_merged_df = [
  #'id', # Keep 'id' for debugging purposes
  'city',
  'date',
  'toss_winner',
  'toss_decision',
  'win_by_wickets',
  'extra_runs',
  # 'id'
]
merged_df = merged_df.drop(columns=columns_to_drop_from_merged_df, errors='ignore')

print("DataFrame after dropping specified columns:")
merged_df_covariates = merged_df
display(merged_df)

# **Rank Difference Previous Year**

In [ ]:
RANK_LISTED_RAW = {
    2008: [2, 7, 3, 1, 5, 4, 0, 6],
    2009: [1, 3, 0, 4, 7, 6, 5, 2],
    2010: [2, 1, 4, 7, 5, 0, 6, 3],
    2011: [1, 6, 9, 4, 7, 3, 2, 8, 5, 0],
    2012: [3, 7, 0, 5, 1, 2, 8, 6, 4],
    2013: [0, 8, 5, 6, 1, 7, 2, 4, 3],
    2014: [2, 7, 0, 1, 3, 4, 6, 5],
    2015: [0, 6, 7, 4, 1, 3, 2, 5],
    2016: [5, 0, 7, 3, 4, 6, 1, 2],
    2017: [5, 6, 4, 3, 0, 1, 7, 2],
    2018: [1, 7, 5, 2, 4, 3, 6, 0],
    2019: [1, 2, 5, 4, 0, 6, 7, 3]}

def get_team_mappings_for_year(df_source, year):
  ipl_year_df_temp = df_source[df_source['year'] == year].copy()
  if ipl_year_df_temp.empty:
      return {}, {}

  unique_teams_for_year = sorted(list(pd.concat([ipl_year_df_temp['team1'], ipl_year_df_temp['team2']]).unique()))
  team_to_id = {team: i for i, team in enumerate(unique_teams_for_year)}
  id_to_team = {i: team for team, i in team_to_id.items()}
  return team_to_id, id_to_team


all_teams_to_id = {}
for year in sorted(merged_df['year'].unique()):
  team_to_id_for_year, _ = get_team_mappings_for_year(merged_df, year)
  all_teams_to_id[year] = team_to_id_for_year


merged_df_covariates.loc[:, 'rank_difference_prev_year'] = 0.0
debug_count = 0
MAX_DEBUG_PRINTS = 10

for idx, row in merged_df_covariates.iterrows():
    current_year = row['year']
    previous_year = current_year - 1

    team1_name = row['team1']
    team2_name = row['team2']

    rank_diff = 0

    if previous_year in all_teams_to_id and previous_year in RANK_LISTED_RAW:
        team_to_id_prev_year = all_teams_to_id[previous_year]
        true_rank_prev_year = RANK_LISTED_RAW[previous_year]

        if team_to_id_prev_year and true_rank_prev_year:
            team1_id = team_to_id_prev_year.get(team1_name)
            team2_id = team_to_id_prev_year.get(team2_name)

            if team1_id is not None and team2_id is not None and team1_id in true_rank_prev_year and team2_id in true_rank_prev_year:
                rank_t1 = true_rank_prev_year.index(team1_id)
                rank_t2 = true_rank_prev_year.index(team2_id)
                rank_diff = rank_t1 - rank_t2

            if rank_diff != 0 and debug_count < MAX_DEBUG_PRINTS:
                print(f"Match ID: {row['id']}, Year: {current_year}, Teams: {team1_name} vs {team2_name}")
                print(f"  Prev Ranks ({previous_year}): {team1_name} (ID {team1_id}): Rank {rank_t1}, {team2_name} (ID {team2_id}): Rank {rank_t2}")
                print(f"  Calculated Diff: {rank_diff * -1}")
                debug_count += 1

    merged_df_covariates.loc[idx, 'rank_difference_prev_year'] = rank_diff * -1

print(f"\nDisplaying a sample of merged_df_covariates with 'rank_difference_prev_year':")
display(merged_df_covariates.head())


# **Rank Scores Produced**

In [ ]:
def rank_scores(scores):
  rank = {i:scores[i] for i in range(len(scores))}
  rank = sorted(rank, key=lambda x: -rank[x])

  final_rank = [0]*len(rank)
  for i in range(len(rank)):
    rk = i
    team = rank[i]
    final_rank[team] = rk
  return final_rank


rank_scores([ 1.45914145e-02, -5.47620630e+00,  6.89643731e-01,  3.94833692e+00,
        -3.98612284e+00,  1.02035632e+01,  1.44888135e+01, -1.98826197e+01])